In [ ]:
import os
import chemiscope
import ipi
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
import nqetools as nqe
# This follows:
# https://atomistic-cookbook.org/examples/path-integrals/path-integrals.html

In [ ]:
# Make a directory to store everything
directory_md = "md"
directory_piglet = "piglet"
directory_pimd = "pimd"
n_beads = 8
timestep = 0.5
total_steps = 200
stride = 1
temperature = 298

In [ ]:
# build the molecule
atoms = read("water_32.pdb")
atoms.center(vacuum=0.0)
# Set PBC
atoms.set_pbc([True, True, True])

In [ ]:
# Make sure the directory is empty
nqe.remove_directory(directory_pimd)
# Run the calculation
nqe.run_md(directory_pimd, atoms,
           driver='ase-mace',
           md_type="NVT",
           xml_in="input_pimd.xml",
           n_beads=n_beads,
           timestep=timestep,
           total_steps=total_steps,
           stride=stride,
           temperature=temperature)

In [ ]:
# Read the results
# drops first frame where all atoms overlap
output_data, output_desc = ipi.read_output(os.path.join(directory_pimd, "md.out"))
traj_data = [ipi.read_trajectory(os.path.join(directory_pimd, f"md.pos_{i}.xyz"))[1:] for i in range(n_beads)]

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=(4, 3), constrained_layout=True)
ax.plot(
    output_data["time"],
    output_data["potential"] - output_data["potential"][0],
    "b-",
    label="Potential, $V$",
)
ax.plot(
    output_data["time"],
    output_data["conserved"] - output_data["conserved"][0],
    "r-",
    label="Conserved, $H$",
)
ax.set_xlabel(r"$t$ / ps")
ax.set_ylabel(r"energy / eV")
ax.legend()
plt.show()
# https://github.com/lab-cosmo/chemiscope/issues/390

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=(4, 3), constrained_layout=True)
ax.plot(
    output_data["time"],
    output_data["kinetic_cv"],
    "b-",
    label="Centroid virial, $K_{CV}$",
)
ax.plot(
    output_data["time"],
    output_data["kinetic_td"],
    "r-",
    label="Thermodynamic, $K_{TD}$",
)
ax.set_xlabel(r"$t$ / ps")
ax.set_ylabel(r"energy / eV")
ax.legend()
plt.show()

In [ ]:
traj_pimd = chemiscope.ase_merge_pi_frames(traj_data)
# we also tweak the visualization options, and then show the viewer
traj_pimd["shapes"]["paths"]["parameters"]["global"]["radius"] = 0.05
traj_pimd["settings"]["structure"][0].update(
    dict(
        atoms=False,
        keepOrientation=True,
        color={"property": "bead_id", "palette": "hsv (periodic)"},
    )
)

chemiscope.show(**traj_pimd, mode="structure")

In [ ]:
# Make sure the directory is empty
nqe.remove_directory(directory_piglet)
# Run the calculation
nqe.run_md(directory_piglet, atoms,
           driver='ase-mace',
           md_type="NVT",
           xml_in="input_piglet.xml",
           n_beads=n_beads,
           timestep=0.5,
           total_steps=200,
           stride=1,
           temperature=298)

In [ ]:
# drops first frame
output_gle, desc_gle = ipi.read_output(os.path.join(directory_piglet, "md.out"))
traj_gle = [ipi.read_trajectory(os.path.join(directory_piglet, f"md.pos_{i}.xyz"))[1:] for i in range(8)]

fig, ax = plt.subplots(1, 1, figsize=(4, 3), constrained_layout=True)
ax.plot(
    output_data["time"],
    output_data["potential"] - output_data["potential"][0],
    "b--",
    label="PIMD",
)
ax.plot(
    output_gle["time"],
    output_gle["potential"] - output_gle["potential"][0],
    "b-",
    label="PIGLET",
)
ax.set_xlabel(r"$t$ / ps")
ax.set_ylabel(r"energy / eV")
ax.legend()
plt.show()

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=(4, 3), constrained_layout=True)
ax.plot(output_data["time"], output_data["kinetic_cv"], "b--", label="PIMD, $K_{CV}$")
ax.plot(output_gle["time"], output_gle["kinetic_cv"], "b", label="PIGLET, $K_{CV}$")
ax.plot(output_data["time"], output_data["kinetic_td"], "r--", label="PIMD, $K_{TD}$")
ax.plot(output_gle["time"], output_gle["kinetic_td"], "r", label="PIGLET, $K_{TD}$")
ax.set_xlabel(r"$t$ / ps")
ax.set_ylabel(r"energy / eV")
ax.legend()
plt.show()

In [ ]:
kinetic_cv = ipi.read_trajectory(os.path.join(directory_piglet, "md.kin.xyz"))[1:]
kinetic_od = ipi.read_trajectory(os.path.join(directory_piglet,"md.kod.xyz"))[1:]
kinetic_tens = np.hstack(
    [
        np.asarray([k.positions for k in kinetic_cv[-10:]]).mean(axis=0),
        np.asarray([k.positions for k in kinetic_od[-10:]]).mean(axis=0),
    ]
)

centroid = traj_gle[-1][-1].copy()
centroid.positions = np.asarray([t[-1].positions for t in traj_gle]).mean(axis=0)
centroid.arrays["kinetic_cv"] = kinetic_tens

In [ ]:
ellipsoids = chemiscope.ase_tensors_to_ellipsoids(
    [centroid], "kinetic_cv", scale=15, force_positive=True
)

chemiscope.show(
    [centroid],
    shapes={"kinetic_cv": ellipsoids},
    mode="structure",
    settings=chemiscope.quick_settings(
        structure_settings={
            "shape": ["kinetic_cv"],
            "unitCell": True,
        }
    ),
)